# 🌸 Hannah – Agente de QA con IA

Este notebook demuestra cómo Hannah transforma un requerimiento en una matriz de pruebas y casos Gherkin.

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import pandas as pd

# Cargar variables de entorno
load_dotenv()
client = OpenAI()

In [ ]:
# Ejemplo de few-shot
few_shot_example = """
Ejemplo de matriz de prueba:

| ID    | Escenario                  | Datos de entrada        | Resultado esperado               |
|-------|----------------------------|-------------------------|----------------------------------|
| TC001 | Login exitoso              | Usuario válido          | Acceso concedido                 |
| TC002 | Login fallido              | Usuario inválido        | Mensaje de error                 |

Ejemplo de casos Gherkin:

Feature: Validar login de usuarios

  Scenario: Usuario válido accede con credenciales correctas
    Given un usuario registrado
    When introduce usuario y contraseña válidos
    Then debe acceder exitosamente al sistema
"""

In [ ]:
# Requerimiento de ejemplo
requerimiento = """
Titulo: [IOS] Generar acceso en menú hamburguesa con la etiqueta nuevo
Objetivo: Validar que solo clientes Premier con flag activo vean la opción.
"""

In [ ]:
# Clase Hannah 🌸
class Hannah:
    def __init__(self, client):
        self.client = client

    def generar_pruebas(self, requerimiento, few_shot_example):
        resp = self.client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "Eres Hannah 🌸, un agente de QA que genera matrices de prueba y casos Gherkin."},
                {"role": "user", "content": f"""Ejemplo:\n{few_shot_example}\n\nRequerimiento:\n{requerimiento}"""}
            ],
            temperature=0.3
        )
        return resp.choices[0].message.content

In [ ]:
# Crear instancia y generar pruebas
hannah = Hannah(client)
output = hannah.generar_pruebas(requerimiento, few_shot_example)
print(output)

In [ ]:
# Procesar tabla y exportar resultados
rows = []
for line in output.splitlines():
    if "|" in line and "---" not in line:
        cols = [c.strip() for c in line.split("|") if c.strip()]
        if len(cols) > 1:
            rows.append(cols)

if rows:
    df = pd.DataFrame(rows[1:], columns=rows[0])
    df.to_excel("matriz_pruebas.xlsx", index=False)
    display(df)

gherkin = []
capture = False
for line in output.splitlines():
    if line.strip().startswith(("Feature", "Scenario")):
        capture = True
    if capture:
        gherkin.append(line)

if gherkin:
    with open("casos.feature", "w", encoding="utf-8") as f:
        f.write("\n".join(gherkin))
    print("✅ Archivos generados: matriz_pruebas.xlsx y casos.feature")